In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


# Import Libraries

In [2]:
pip install sentence-transformers transformers datasets accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 96.2 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [3]:
import pandas as pd
import numpy as np
import string

from transformers import AutoTokenizer, AutoModelForMultipleChoice,TrainingArguments, Trainer
from datasets import Dataset


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# EDA

In [4]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

print(train.shape)
print(test.shape)

(2000, 8)
(500, 7)


In [5]:
train.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

In [7]:
train.isnull().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
dtype: int64

# Evaluation Metric

In [8]:
def map3(actuals, predictions):
    score = 0

    for actual, pred in zip(actuals, predictions):
        if actual == pred[0]:
            score += 1
        elif actual == pred[1]:
            score += 1/2
        elif actual == pred[2]:
            score += 1/3

    return score / len(actuals)

In [9]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


# Model 2 (Pretrained): DeBERTa

In [10]:
train_questions, val_questions = train_test_split(
    train,
    test_size=0.2,
    stratify=train["answer"],
    random_state=4524
)

print(train_questions.shape)
print(val_questions.shape)

(1600, 8)
(400, 8)


In [11]:
model_name = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [12]:
def preprocess_multiple_choice(examples):
    # Repeat the prompt 5 times for the 5 choices
    first_sentences = [[prompt] * 5 for prompt in examples["prompt"]]
    
    # Grab options A through E for each question
    choices = ["A", "B", "C", "D", "E"]
    second_sentences = [
        [examples[choice][i] for choice in choices]
        for i in range(len(examples["prompt"]))
    ]
    
    # Flatten both lists to pass to the tokenizer
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    # Tokenize pairs
    tokenized = tokenizer(
        first_sentences, 
        second_sentences, 
        truncation=True, 
        max_length=384, 
        padding="max_length"
    )
    
    # Unflatten back into the shape: (batch_size, num_choices, seq_len)
    return {k: [v[i : i + 5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

# Convert string answers (A-E) to integers (0-4)
choice_map = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

# Map directly from your original splits
train_questions["label"] = train_questions["answer"].map(choice_map)
val_questions["label"] = val_questions["answer"].map(choice_map)

train_ds = Dataset.from_pandas(train_questions)
val_ds = Dataset.from_pandas(val_questions)

# Tokenize using the new function
train_ds = train_ds.map(preprocess_multiple_choice, batched=True)
val_ds = val_ds.map(preprocess_multiple_choice, batched=True)

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [13]:
# UPDATE THIS CELL TO MATCH THE NEW STRUCTURE
train_ds = train_ds.remove_columns(["prompt", "A", "B", "C", "D", "E", "answer"])
val_ds = val_ds.remove_columns(["prompt", "A", "B", "C", "D", "E", "answer"])

# Note the column name 'label' here (no 's')
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
val_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [14]:
model = AutoModelForMultipleChoice.from_pretrained(model_name,torch_dtype=torch.float32)
model = model.cuda()

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/371M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                  

model.safetensors:   0%|          | 0.00/371M [00:00<?, ?B/s]

In [15]:
training_args = TrainingArguments(
    output_dir="./deberta_base",
    learning_rate=1e-5,
    per_device_train_batch_size=2,  
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  
    gradient_checkpointing=True,    
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    fp16=True,                      
    bf16=False,
    report_to="none"
)

In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

In [17]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss
1,No log,3.172931
2,No log,2.699216
3,No log,2.195203
4,No log,1.917725
5,No log,1.832779


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=21.9319296875, metrics={'train_runtime': 2941.8288, 'train_samples_per_second': 2.719, 'train_steps_per_second': 0.085, 'total_flos': 7893402347520000.0, 'train_loss': 21.9319296875, 'epoch': 5.0})

In [18]:
choices = ["A", "B", "C", "D", "E"]

def predict_question(row):
    # Format inputs exactly like training
    first_sentences = [row["prompt"]] * 5
    second_sentences = [row[c] for c in choices]
    
    # Match the max_length padding used during training
    encodings = tokenizer(
        first_sentences,
        second_sentences,
        padding="max_length",
        truncation=True,
        max_length=384,
        return_tensors="pt"
    )
    
    # Reshape tensors to add the batch dimension: (1, 5, seq_len)
    encodings = {k: v.unsqueeze(0).to(model.device) for k, v in encodings.items()}
    
    with torch.no_grad():
        outputs = model(**encodings)
        # Outputs shape is (1, 5). Grab the logits directly.
        logits = outputs.logits.detach().cpu().numpy()[0]
        
    # Sort options by highest logit score descending
    order = np.argsort(logits)[::-1]
    return [choices[i] for i in order[:3]]

In [19]:
val_predictions = []

for _, row in val_questions.iterrows():

    val_predictions.append(
        predict_question(row)
    )

score = map3(
    val_questions["answer"].tolist(),
    val_predictions
)

print("Validation MAP@3:", score)

Validation MAP@3: 0.7750000000000001


In [20]:
test_predictions = []

for _, row in test.iterrows():

    test_predictions.append(
        " ".join(
            predict_question(row)
        )
    )

# Submission Cell

In [21]:
submission = pd.DataFrame({
    "ID": test["id"],
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

submission.head()

,ID,Prediction
0,1,C A B
1,2,B C D
2,3,E B D
3,4,E C A
4,5,D E A
